# Hazard Waste Detection — Colab GPU Training

Edit code in **VS Code**, run training here on a **Colab NVIDIA GPU** (T4/L4/A100).

**Do not use TPU** — YOLOv9 segmentation here is PyTorch/CUDA.

## Setup options
- **Option A:** Project + dataset on Google Drive
- **Option B:** Clone project from GitHub, dataset on Drive

Same entry point as local training: `08_train_yolov9.py`

In [ ]:
# 1) Mount Drive and set paths
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

# CHANGE THIS to your Drive folder
PROJECT_ROOT = Path('/content/drive/MyDrive/Hazard_Waste_Detection')
DATASET_ROOT = PROJECT_ROOT / 'hazard_dataset_clean'

os.environ['HAZARD_DATASET_ROOT'] = str(DATASET_ROOT)
%cd {PROJECT_ROOT}

assert DATASET_ROOT.exists(), f'Missing dataset: {DATASET_ROOT}'
print('Project:', PROJECT_ROOT)
print('Dataset:', DATASET_ROOT)

In [ ]:
# 2) GPU check (must be NVIDIA CUDA)
!nvidia-smi

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No CUDA GPU. Runtime -> Change runtime type -> GPU (not TPU).')

In [ ]:
# 3) Install dependencies (first run only)
!pip install -q -r requirements-train.txt
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124

# Clone YOLOv9 if not bundled in Drive copy
if not Path('yolov9/segment/train.py').exists():
    !git clone --depth 1 https://github.com/WongKinYiu/yolov9.git yolov9

In [ ]:
# 4) Fresh training (100 epochs)
!python 08_train_yolov9.py --device 0 --epochs 100 --batch-size 16

In [ ]:
# 4b) OR resume from local checkpoint (upload last.pt + results.csv to Drive first)
# Copy local files to:
#   yolov9/runs/train-seg/hazard_waste_seg/weights/last.pt
#   yolov9/runs/train-seg/hazard_waste_seg/results.csv
!python 08_train_yolov9.py --device 0 --epochs 100 --batch-size 16 --resume auto

In [ ]:
# 5) Evaluation only (if training finished elsewhere)
!python 08_train_yolov9.py --device 0 --skip-train --batch-size 16

## After training

Download back to VS Code:
- `evaluation_reports/final_evaluation_report.txt`
- `yolov9/runs/train-seg/hazard_waste_seg/weights/best.pt`
- `yolov9/runs/train-seg/hazard_waste_seg/results.csv`
- curve PNGs in `evaluation_reports/`

Paste `final_evaluation_report.txt` into your Task 5 submission.